# SIA - Model Training

Loads pseudo-labels from the EDA notebook and fine-tunes DistilBERT for mismatch classification.

In [1]:
import os, re, json, pickle, warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings('ignore')

P2N = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
EXP_HRS = {'Low': 72, 'Medium': 24, 'High': 8, 'Critical': 2}

In [2]:
df = pd.read_csv('outputs/pseudo_labels.csv')
print(df.shape)
print(df['mismatch'].value_counts())

(20000, 29)
mismatch
0    12533
1     7467
Name: count, dtype: int64


## Build input features for the classifier

In [3]:
# include priority + resolution time so the model can learn the mismatch pattern
# e.g. Priority=Low + Resolution=under 12h is anomalous

def res_bin(h):
    if h < 12:
        return 'under 12h'
    elif h < 36:
        return '12 to 36h'
    elif h < 72:
        return '36 to 72h'
    elif h < 120:
        return '72 to 120h'
    return 'over 120h'

df['clf_input'] = (
    'Subject: ' + df['subject_clean'] + ' . ' +
    'Details: ' + df['desc_clean'] + ' . ' +
    'Priority: ' + df['Priority_Level'] + ' . ' +
    'Resolution: ' + df['res_hours'].apply(res_bin) + ' . ' +
    'Channel: ' + df['channel'] + ' . ' +
    'Category: ' + df['category'] + ' . ' +
    'Tier: ' + df['domain_tier']
)

print(df['clf_input'].iloc[0])

Subject: hours of operation - individual . Details: hi support, where is your headquarters located? lay soon message show know main. . Priority: High . Resolution: 36 to 72h . Channel: web form . Category: General Inquiry . Tier: consumer


## Train/val/test split

In [4]:
texts = df['clf_input'].tolist()
labels = df['mismatch'].tolist()

X_tr, X_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.15, random_state=42, stratify=labels)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.15, random_state=42, stratify=y_tr)

print(f"train: {len(X_tr)}, val: {len(X_va)}, test: {len(X_te)}")

train: 14450, val: 2550, test: 3000


## Fine-tune DistilBERT

Freezing the first 4 transformer layers and only training the top 2 + classification head to avoid overfitting on pseudo-labeled data.

In [5]:
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print(device)

EPOCHS = 3
BATCH = 16
MAXLEN = 256
LR = 5e-6

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class TicketDS(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], max_length=MAXLEN, padding='max_length',
                        truncation=True, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[i], dtype=torch.long)
        }

mps


In [6]:
# handle class imbalance with weighted sampler
cc = np.bincount(y_tr)
sample_weights = [1.0 / cc[l] for l in y_tr]
sampler = WeightedRandomSampler(sample_weights, len(y_tr), replacement=True)

tr_dl = DataLoader(TicketDS(X_tr, y_tr), batch_size=BATCH, sampler=sampler)
va_dl = DataLoader(TicketDS(X_va, y_va), batch_size=BATCH, shuffle=False)
te_dl = DataLoader(TicketDS(X_te, y_te), batch_size=BATCH, shuffle=False)

In [7]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.to(device)

# freeze layers 0-3, train layers 4-5 + head
for name, param in model.named_parameters():
    if not any(x in name for x in ['layer.4', 'layer.5', 'pre_classifier', 'classifier']):
        param.requires_grad = False

print(f"trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float).to(device))

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01
)
total_steps = len(tr_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 14,767,874


In [8]:
def evaluate(loader):
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for b in loader:
            out = model(input_ids=b['input_ids'].to(device),
                        attention_mask=b['attention_mask'].to(device))
            preds.extend(out.logits.argmax(-1).cpu().tolist())
            true.extend(b['label'].tolist())
    return preds, true

best_f1 = 0
best_weights = None

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    for i, b in enumerate(tr_dl):
        optimizer.zero_grad()
        out = model(input_ids=b['input_ids'].to(device),
                    attention_mask=b['attention_mask'].to(device))
        loss = criterion(out.logits, b['label'].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        if (i + 1) % 100 == 0:
            print(f"epoch {epoch+1} step {i+1}/{len(tr_dl)} loss {running_loss/(i+1):.4f}")

    preds, true = evaluate(va_dl)
    f1 = f1_score(true, preds, average='macro')
    acc = accuracy_score(true, preds)
    print(f"epoch {epoch+1} — val acc: {acc:.4f}  f1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

epoch 1 step 100/904 loss 0.6959
epoch 1 step 200/904 loss 0.6758
epoch 1 step 300/904 loss 0.6646
epoch 1 step 400/904 loss 0.6364
epoch 1 step 500/904 loss 0.5879
epoch 1 step 600/904 loss 0.5540
epoch 1 step 700/904 loss 0.5199
epoch 1 step 800/904 loss 0.4929
epoch 1 step 900/904 loss 0.4661
epoch 1 — val acc: 0.8898  f1: 0.8865
epoch 2 step 100/904 loss 0.2332
epoch 2 step 200/904 loss 0.2309
epoch 2 step 300/904 loss 0.2374
epoch 2 step 400/904 loss 0.2351
epoch 2 step 500/904 loss 0.2321
epoch 2 step 600/904 loss 0.2297
epoch 2 step 700/904 loss 0.2260
epoch 2 step 800/904 loss 0.2260
epoch 2 step 900/904 loss 0.2237
epoch 2 — val acc: 0.9020  f1: 0.8995
epoch 3 step 100/904 loss 0.2087
epoch 3 step 200/904 loss 0.1962
epoch 3 step 300/904 loss 0.1967
epoch 3 step 400/904 loss 0.1931
epoch 3 step 500/904 loss 0.1912
epoch 3 step 600/904 loss 0.1894
epoch 3 step 700/904 loss 0.1904
epoch 3 step 800/904 loss 0.1903
epoch 3 step 900/904 loss 0.1914
epoch 3 — val acc: 0.9035  f1: 0.

In [9]:
# test set evaluation
model.load_state_dict(best_weights)
model.to(device)

preds, true = evaluate(te_dl)
print(classification_report(true, preds, target_names=['Consistent', 'Mismatch']))
print(f"accuracy: {accuracy_score(true, preds):.4f}")
print(f"macro f1: {f1_score(true, preds, average='macro'):.4f}")

              precision    recall  f1-score   support

  Consistent       1.00      0.83      0.91      1880
    Mismatch       0.78      0.99      0.87      1120

    accuracy                           0.89      3000
   macro avg       0.89      0.91      0.89      3000
weighted avg       0.92      0.89      0.90      3000

accuracy: 0.8933
macro f1: 0.8909


In [10]:
model.save_pretrained('models/distilbert_mismatch')
tokenizer.save_pretrained('models/distilbert_mismatch')
print("saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved


## Sample dossiers

Generate a few example dossiers to check the format.

In [11]:
CRISIS_SHORT = ['outage', 'down', 'breach', 'not working', 'cannot access',
               'crash', 'locked out', 'compromised', 'fraud', 'emergency']

def make_analysis(info):
    try:
        import requests
        r = requests.post('http://localhost:11434/api/generate',
                          json={'model': 'mistral', 'stream': False,
                                'prompt': (
                                    'You are a support ticket auditor. Write 2-3 sentences '
                                    'explaining why this ticket has a priority mismatch. '
                                    'Only use these facts and nothing else:\n'
                                    f"Subject: {info['subject']}\n"
                                    f"Description: {info['desc']}\n"
                                    f"Channel: {info['channel']}\n"
                                    f"Resolution time: {info['res_hours']:.0f} hours\n"
                                    f"Assigned: {info['assigned']}\n"
                                    f"Inferred: {info['inferred']}")},
                          timeout=30)
        if r.status_code == 200:
            return r.json()['response'].strip()
    except:
        pass
    direction = 'underestimated' if info['delta'] > 0 else 'overestimated'
    exp = EXP_HRS.get(info['assigned'], 24)
    return (f"The assigned priority '{info['assigned']}' appears {direction}. "
            f"Resolution took {info['res_hours']:.0f}h (expected ~{exp}h for "
            f"'{info['assigned']}' tickets) via {info['channel']}.")

In [12]:
# get predictions on test set
model.eval()
all_preds, all_confs = [], []
with torch.no_grad():
    for i in range(0, len(X_te), BATCH):
        enc = tokenizer(X_te[i:i+BATCH], max_length=MAXLEN, padding=True,
                        truncation=True, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        probs = torch.softmax(out.logits, -1)
        all_preds.extend(out.logits.argmax(-1).cpu().tolist())
        all_confs.extend(probs[:, 1].cpu().tolist())

test_df = df[df['clf_input'].isin(set(X_te))].iloc[:len(X_te)].reset_index(drop=True)

dossiers = []
for i, (pred, conf) in enumerate(zip(all_preds, all_confs)):
    if pred != 1 or i >= len(test_df):
        continue
    if len(dossiers) >= 20:
        break

    row = test_df.iloc[i]
    text = row['subject_clean'] + ' ' + row['desc_clean']
    rt = float(row['res_hours'])
    exp = EXP_HRS.get(row['Priority_Level'], 24)
    ratio = rt / max(exp, 0.1)

    if ratio > 2:
        interp = f"{rt:.0f}h is {ratio:.1f}x above expected {exp}h for {row['Priority_Level']}"
    elif ratio < 0.4:
        interp = f"{rt:.0f}h is much faster than expected {exp}h for {row['Priority_Level']}"
    else:
        interp = f"{rt:.0f}h is within normal range for {row['Priority_Level']}"

    kw_ev = [{'signal': 'keyword', 'value': kw, 'weight': str(round(min(0.9, 0.3 + text.count(kw) * 0.15), 2))}
             for kw in CRISIS_SHORT if kw in text][:3]

    dossiers.append({
        'ticket_id': row['ticket_id'],
        'assigned_priority': row['Priority_Level'],
        'inferred_severity': row['inferred'],
        'mismatch_type': row['mismatch_type'],
        'severity_delta': f"{int(row['delta']):+d}",
        'feature_evidence': kw_ev + [
            {'signal': 'resolution_time', 'value': f'{rt:.0f}h', 'interpretation': interp},
            {'signal': 'channel', 'value': row['channel'], 'interpretation': f"submitted via {row['channel']}"}
        ],
        'constraint_analysis': make_analysis({
            'subject': row['subject_clean'], 'desc': row['desc_clean'],
            'channel': row['channel'], 'res_hours': rt,
            'assigned': row['Priority_Level'], 'inferred': row['inferred'],
            'delta': int(row['delta'])
        }),
        'confidence': str(round(conf, 3))
    })

with open('outputs/dossiers.json', 'w') as f:
    json.dump(dossiers, f, indent=2)

print(f"generated {len(dossiers)} dossiers")
print(json.dumps(dossiers[0], indent=2))

generated 20 dossiers
{
  "ticket_id": "TKT-100006",
  "assigned_priority": "Medium",
  "inferred_severity": "Low",
  "mismatch_type": "False Alarm",
  "severity_delta": "-1",
  "feature_evidence": [
    {
      "signal": "resolution_time",
      "value": "106h",
      "interpretation": "106h is 4.4x above expected 24h for Medium"
    },
    {
      "signal": "channel",
      "value": "chat",
      "interpretation": "submitted via chat"
    }
  ],
  "constraint_analysis": "The priority mismatch in this ticket is due to the subject line indicating a password reset request, while the description pertains to upgrading to the enterprise plan and an unclear impact on 'edge wall'. Additionally, the long resolution time of 106 hours suggests that the current priority level (Medium) may not adequately reflect the urgency required for addressing this customer inquiry.",
  "confidence": "0.979"
}
